In [1]:
from pyiceberg.catalog import load_catalog
import duckdb

PROJECT = "/Users/Shared/academy/learner/projects/ethereum_iceberg"
catalog = load_catalog(
    "local",
    type="sql",
    uri=f"sqlite:///{PROJECT}/catalog/pyiceberg_catalog.db",
    warehouse=f"file://{PROJECT}/warehouse",
)
S3 = "s3://aws-public-blockchain/v1.0/eth"
con = duckdb.connect()
con.execute("set s3_region = 'us-east-2'")
catalog.list_namespaces()

[('raw',), ('ref',)]

In [2]:
blocks = catalog.load_table("raw.blocks")
for s in blocks.snapshots():
    print(s.snapshot_id, s.parent_snapshot_id, s.summary.operation, s.summary.get("added-records"), s.summary.get("total-records"))

7781070899906191404 None Operation.APPEND 7177 7177
3859929962084802291 7781070899906191404 Operation.APPEND 215235 222412
917221300865763016 3859929962084802291 Operation.APPEND 215033 437445
7582631125077599049 917221300865763016 Operation.APPEND 222451 659896


In [3]:
nov1 = con.execute(f"select * exclude (logs_bloom, last_modified) from read_parquet('{S3}/blocks/date=2026-11-01/*.parquet')").to_arrow_table()
blocks.append(nov1)
blocks.current_snapshot().summary.additional_properties

{'added-files-size': '1454923',
 'added-data-files': '1',
 'added-records': '7176',
 'total-data-files': '5',
 'total-delete-files': '0',
 'total-records': '667072',
 'total-files-size': '131356844',
 'total-position-deletes': '0',
 'total-equality-deletes': '0'}